# Notebook 01 — Knowledge Graphs & ETL Pipeline

Build a medical knowledge graph from scratch: schema design, Cypher queries, batch ingestion, and LLM-powered entity extraction.



In [ ]:
import sys
from pathlib import Path
# Works from the repository root, experiment directory, or notebooks directory.
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((candidate for ancestor in _candidates
                     for candidate in (ancestor, ancestor / "experiments/graphrag")
                     if (candidate / "src/graphrag/config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter from this repository or its graphrag directory")
sys.path.insert(0, str(PROJECT_ROOT.parents[1]))
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")
load_dotenv(PROJECT_ROOT.parents[1] / ".env")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from graphrag import validate, dump, get_neo4j_client, get_general_llm, get_strong_llm, get_embeddings
validate(require_llm=False)

## 1. Connect to Neo4j

In [ ]:
client = get_neo4j_client()
if not client.verify_connectivity():
    client.close()
    raise ConnectionError("Neo4j is unavailable. Check the configured URI and credentials.")

## 2. Graph Schema Design

Our domain: **Drug Interactions**. We model 4 entity types and 5 relationship types.

In [ ]:
NODE_LABELS = {
    "Drug": "A pharmacological agent (e.g., Warfarin, Aspirin)",
    "Condition": "A medical condition or disease (e.g., Diabetes, Hypertension)",
    "SideEffect": "An adverse effect caused by a drug (e.g., Bleeding, Nausea)",
    "DrugClass": "A pharmacological class (e.g., NSAID, Anticoagulant, SSRI)",
}

RELATIONSHIP_TYPES = {
    "INTERACTS_WITH": "Drug-Drug interaction (bidirectional in meaning, stored as directed edge)",
    "TREATS": "Drug treats a Condition",
    "CAUSES_SIDE_EFFECT": "Drug causes a SideEffect",
    "CONTRAINDICATED_FOR": "Drug is contraindicated for a Condition",
    "BELONGS_TO_CLASS": "Drug belongs to a DrugClass",
}

ALLOWED_REL_TYPES = set(RELATIONSHIP_TYPES.keys())

print("Node Labels:")
for label, desc in NODE_LABELS.items():
    print(f"  :{label} — {desc}")
print("\nRelationship Types:")
for rel, desc in RELATIONSHIP_TYPES.items():
    print(f"  [{rel}] — {desc}")

## 3. Constraints, Indexes & Sample Data

In [ ]:
from graphrag.schema import CONSTRAINT_QUERIES, INDEX_QUERIES, ensure_schema
ensure_schema(client)
# Optional sample additions. Default execution preserves existing graph data.
RUN_SAMPLE_DEMO = False
sample_cypher = """
MERGE (warfarin:Drug {name: 'Warfarin'})
MERGE (aspirin:Drug {name: 'Aspirin'})
MERGE (ibuprofen:Drug {name: 'Ibuprofen'})
MERGE (bleeding:SideEffect {name: 'Increased Bleeding Risk'})
MERGE (dvt:Condition {name: 'Deep Vein Thrombosis'})
MERGE (anticoag:DrugClass {name: 'Anticoagulant'})
MERGE (nsaid:DrugClass {name: 'NSAID'})

MERGE (warfarin)-[:INTERACTS_WITH {severity: 'Major', mechanism: 'Additive anticoagulant effect'}]->(aspirin)
MERGE (warfarin)-[:INTERACTS_WITH {severity: 'Major', mechanism: 'Increased bleeding via platelet and coagulation pathway'}]->(ibuprofen)
MERGE (warfarin)-[:CAUSES_SIDE_EFFECT]->(bleeding)
MERGE (aspirin)-[:CAUSES_SIDE_EFFECT]->(bleeding)
MERGE (warfarin)-[:TREATS]->(dvt)
MERGE (warfarin)-[:BELONGS_TO_CLASS]->(anticoag)
MERGE (aspirin)-[:BELONGS_TO_CLASS]->(nsaid)
MERGE (ibuprofen)-[:BELONGS_TO_CLASS]->(nsaid)
"""

if RUN_SAMPLE_DEMO:
    client.write(sample_cypher)
    print("Sample data upserted")
else:
    print("Schema ready. Sample additions disabled; full dataset is loaded below.")

> **Try in Neo4j Browser** — paste these queries to see the sample data:
>
> ```cypher
> -- See all nodes and relationships
> MATCH (n)-[r]->(m) RETURN n, r, m
> ```
>
> ```cypher
> -- Check constraints
> SHOW CONSTRAINTS
> ```
>
> ```cypher
> -- Check indexes
> SHOW INDEXES
> ```

## 4. Cypher Queries — Hands-On

In [ ]:
# Find all drugs
results = client.query("MATCH (d:Drug) RETURN d.name AS name, d.description AS desc")
print("All Drugs:")
for r in results:
    print(f"  {r['name']}: {r['desc']}")

# Find what Warfarin interacts with
print("\nWarfarin interactions:")
results = client.query("""
    MATCH (d:Drug {name: 'Warfarin'})-[r:INTERACTS_WITH]-(other:Drug)
    RETURN other.name AS drug, r.severity AS severity, r.mechanism AS mechanism
""")
for r in results:
    print(f"  {r['drug']} — Severity: {r['severity']}, Mechanism: {r['mechanism']}")

> **Try in Neo4j Browser:**
>
> ```cypher
> -- All drugs
> MATCH (d:Drug) RETURN d.name AS name, d.description AS desc
> ```
>
> ```cypher
> -- Warfarin interactions with severity and mechanism
> MATCH (d:Drug {name: 'Warfarin'})-[r:INTERACTS_WITH]-(other:Drug)
> RETURN other.name AS drug, r.severity AS severity, r.mechanism AS mechanism
> ```

In [ ]:
drug_name = "Warfarin"
results = client.query(
    "MATCH (d:Drug {name: $name})-[r]->(m) RETURN type(r) AS rel, m.name AS target",
    {"name": drug_name}
)
print(f"All relationships FROM {drug_name}:")
for r in results:
    print(f"  --[{r['rel']}]--> {r['target']}")

> **Try in Neo4j Browser:**
>
> ```cypher
> -- All outgoing relationships from Warfarin
> MATCH (d:Drug {name: 'Warfarin'})-[r]->(m)
> RETURN type(r) AS rel, m.name AS target
> ```
>
> ```cypher
> -- Visual: full neighborhood of Warfarin
> MATCH path = (d:Drug {name: 'Warfarin'})-[r]-(m) RETURN path
> ```

## 5. Load Full Dataset & Build Graph

Now let's load the complete dataset (80 drugs, 244 interactions) and build the full knowledge graph using batch `UNWIND` + `MERGE`.

In [ ]:
import json
from graphrag.ingestion import DrugRecord, InteractionRecord, TextPassage, DrugInteractionDataset, seed_dataset

dataset_path = PROJECT_ROOT / "data/raw/drug_interactions.json"
with dataset_path.open(encoding="utf-8") as f:
    dataset = DrugInteractionDataset.model_validate(json.load(f))
counts = seed_dataset(client, dataset)
print("Upserted:", counts)
print("Existing graph data was preserved. CLI --clean-first explicitly resets a dedicated database.")

> **Try in Neo4j Browser** — verify the full graph:
>
> ```cypher
> -- Node counts by label
> MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count ORDER BY count DESC
> ```
>
> ```cypher
> -- Relationship counts by type
> MATCH ()-[r]->() RETURN type(r) AS relationship, count(r) AS count ORDER BY count DESC
> ```
>
> ```cypher
> -- Severity distribution
> MATCH ()-[r:INTERACTS_WITH]->() RETURN r.severity AS severity, count(r) AS count ORDER BY count DESC
> ```
>
> ```cypher
> -- Visual: entire graph (may be large)
> MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 200
> ```
>
>```cypher
> -- Warfarin full neighborhood (same as pyvis above)
> MATCH path = (d:Drug {name: 'Warfarin'})-[r]-(m) RETURN path
> ```
>
> ```cypher
> -- Top 10 most-connected drugs
> MATCH (d:Drug)-[r]-() RETURN d.name AS drug, count(r) AS connections ORDER BY connections DESC LIMIT 10
> ```
>
> ```cypher
> -- All drugs in the NSAID class
> MATCH (d:Drug)-[:BELONGS_TO_CLASS]->(dc:DrugClass {name: 'NSAID'}) RETURN d.name AS drug
> ```

## 7. LLM-Powered Entity Extraction

For unstructured text, we use GPT-4o with Pydantic structured output to extract entities and relationships automatically.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

# --- Extraction Schemas ---

class ExtractedEntity(BaseModel):
    """An entity extracted from unstructured text."""
    name: str = Field(..., description="Canonical name of the entity (e.g., 'Warfarin', 'Diabetes')")
    entity_type: Literal["Drug", "Condition", "SideEffect", "DrugClass"] = Field(
        ..., description="One of: Drug, Condition, SideEffect, DrugClass"
    )
    aliases: List[str] = Field(
        default_factory=list,
        description="Alternative names or abbreviations (e.g., ['Coumadin'] for Warfarin)",
    )
    description: Optional[str] = Field(
        None, description="Brief description if mentioned in the text"
    )


class ExtractedRelationship(BaseModel):
    """A relationship extracted from unstructured text."""
    source: str = Field(..., description="Name of the source entity")
    source_type: Literal["Drug", "Condition", "SideEffect", "DrugClass"] = Field(..., description="Type of the source entity")
    target: str = Field(..., description="Name of the target entity")
    target_type: Literal["Drug", "Condition", "SideEffect", "DrugClass"] = Field(..., description="Type of the target entity")
    rel_type: Literal["INTERACTS_WITH", "TREATS", "CAUSES_SIDE_EFFECT", "CONTRAINDICATED_FOR", "BELONGS_TO_CLASS"] = Field(
        ...,
        description="One of: INTERACTS_WITH, TREATS, CAUSES_SIDE_EFFECT, CONTRAINDICATED_FOR, BELONGS_TO_CLASS",
    )
    evidence: str = Field(..., description="Quote from the text supporting this relationship")
    # NOTE: We use explicit typed fields instead of `properties: dict` because
    # OpenAI's structured output requires additionalProperties=false on all objects.
    severity: Optional[Literal["Major", "Moderate", "Minor"]] = Field(None, description="Severity for interactions: Major, Moderate, or Minor")
    mechanism: Optional[str] = Field(None, description="Pharmacological mechanism (for interactions)")


class ExtractionResult(BaseModel):
    """Complete extraction result from a text chunk."""
    entities: List[ExtractedEntity] = Field(default_factory=list)
    relationships: List[ExtractedRelationship] = Field(default_factory=list)


# --- Load prompts from central config ---
from graphrag.prompts import EXTRACTION_SYSTEM_PROMPT, EXTRACTION_HUMAN_PROMPT

print("Extraction schemas and prompts defined.")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def extract_from_text(text, llm=None):
    """Extract entities and relationships from a single text chunk."""
    if llm is None:
        llm = get_strong_llm()
    structured_llm = llm.with_structured_output(ExtractionResult)
    prompt = ChatPromptTemplate.from_messages([
        ("system", EXTRACTION_SYSTEM_PROMPT),
        ("human", EXTRACTION_HUMAN_PROMPT),
    ])
    chain = prompt | structured_llm
    return chain.invoke({"text": text})

from graphrag import get_api_key
if dataset.text_passages and get_api_key():
    # Optional demo: extract from first text passage
    sample_text = dataset.text_passages[0].text
    print(f"Input text:\n{sample_text[:300]}...\n")
    result = extract_from_text(sample_text)
    print(f"Extracted {len(result.entities)} entities, {len(result.relationships)} relationships")
    for e in result.entities:
        print(f"  Entity: {e.name} ({e.entity_type})")
    for r in result.relationships:
        print(f"  Rel: {r.source} --[{r.rel_type}]--> {r.target}")
else:
    print("Skipping optional extraction demo: no passage or LLM API key configured.")

> **Try in Neo4j Browser** — if the extracted entities were ingested, you could query them:
>
> ```cypher
> -- Find all drugs that interact with any NSAID
> MATCH (d:Drug)-[:BELONGS_TO_CLASS]->(dc:DrugClass {name: 'NSAID'})
> MATCH (d)-[r:INTERACTS_WITH]-(other:Drug)
> RETURN d.name AS nsaid, other.name AS interacts_with, r.severity
> ```
>
> ```cypher
> -- Drugs that cause bleeding-related side effects
> MATCH (d:Drug)-[:CAUSES_SIDE_EFFECT]->(se:SideEffect)
> WHERE toLower(se.name) CONTAINS 'bleed'
> RETURN d.name AS drug, se.name AS side_effect
> ```

## Summary

| Concept | Implementation |
|---|---|
| Schema Design | 4 node labels, 5 relationship types |
| Cypher | MATCH, WHERE, RETURN, MERGE, parameterized queries |
| UNWIND + MERGE | Batch graph ingestion — fast and idempotent |
| LLM Extraction | GPT-4o + structured output for entity extraction |

**Next:** run the CRAG GraphRAG notebook to exercise the retrieval agent.

In [ ]:
client.close()
print("Done! Neo4j connection closed.")